In [65]:
import pandas as pd
from sklearn.metrics import DistanceMetric

# load data

In [66]:
df = pd.read_csv('/Users/connorhall/datasets/inst414/module 3 assignment/Public.csv', usecols=['SPECIES', 'STATE'])
df = df.dropna()

In [67]:
# remove unknown species
df = df[~df['SPECIES'].str.contains('unknown|perching birds', case=False)]
# species must have >= 50 collisions
df = df[df.groupby('SPECIES')['SPECIES'].transform('count') >= 50]
df

,STATE,SPECIES
8,CA,American robin
11,LA,Blackbirds
13,NY,Gulls
14,MI,Mourning dove
15,CA,American kestrel
...,...,...
339266,IN,Chimney swift
339267,WA,Cedar waxwing
339268,IL,Yellow-rumped warbler
339269,VA,European starling


In [68]:
len(df['SPECIES'].unique())

249

# calculate collision counts

In [69]:
collision_count = pd.crosstab(df['SPECIES'], df['STATE']).stack('STATE')
collision_count

SPECIES            STATE
American barn owl  AB       0
                   AK       0
                   AL       4
                   AR       7
                   AS       3
                           ..
Zebra dove         VT       0
                   WA       0
                   WI       0
                   WV       0
                   WY       0
Length: 16683, dtype: int64

### separate species and state index

In [70]:
species = collision_count.index.get_level_values('SPECIES').tolist()
states = collision_count.index.get_level_values('STATE').tolist()

collisions_df = pd.DataFrame({'Species': species, 'State': states})
collisions_df['Count'] = collision_count.values
collisions_df

,Species,State,Count
0,American barn owl,AB,0
1,American barn owl,AK,0
2,American barn owl,AL,4
3,American barn owl,AR,7
4,American barn owl,AS,3
...,...,...,...
16678,Zebra dove,VT,0
16679,Zebra dove,WA,0
16680,Zebra dove,WI,0
16681,Zebra dove,WV,0


### add collision counts to new dataframe format

In [71]:
# create lists of unique species names and state names
unique_sp = sorted(list(set(species)))
unique_st = sorted(list(set(states)))

In [72]:
# create list of collision count lists
counts = []
for sp in unique_sp:
    sp_filter = collisions_df[collisions_df['Species'] == sp]

    counts.append(sp_filter['Count'].tolist())

In [73]:
# create new dataframe format
collisions_df = pd.DataFrame(counts, columns=unique_st, index=unique_sp)
pd.options.display.max_columns = None
collisions_df

,AB,AK,AL,AR,AS,AZ,BC,CA,CO,CT,DC,DE,FL,FN,GA,GU,HI,IA,ID,IL,IN,KS,KY,LA,MA,MB,MD,ME,MH,MI,MN,MO,MP,MS,MT,NC,ND,NE,NH,NJ,NL,NM,NS,NV,NY,OH,OK,ON,OR,PA,PR,QC,RI,SC,SD,SK,TN,TX,UM,UT,VA,VI,VT,WA,WI,WV,WY
American barn owl,0,0,4,7,3,8,0,783,35,0,14,0,40,4,47,1,399,0,9,8,2,1,6,12,2,0,1,0,0,2,0,11,0,8,0,9,0,1,0,36,0,1,0,5,171,8,5,0,199,3,1,0,0,0,1,0,66,116,0,102,5,0,0,151,0,0,1
American black duck,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,1,0,0,0,0,44,0,0,0,0,0,1,0,0,0,0,0,0,0,0,4,0,0,0,0,38,1,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0
American coot,0,0,0,2,0,29,0,95,8,0,0,0,12,0,5,0,0,0,5,23,5,1,8,2,0,0,0,0,0,1,26,10,0,1,1,1,5,5,0,1,0,0,0,22,21,1,1,0,13,3,1,0,0,1,3,0,9,23,0,49,3,0,0,19,3,1,0
American crow,0,2,5,2,0,1,0,96,9,7,8,0,15,2,16,0,0,4,2,29,10,1,23,3,13,0,7,17,0,16,25,6,0,2,3,18,3,6,7,21,0,1,0,0,58,19,3,0,59,26,0,0,2,4,1,0,25,56,0,4,15,0,14,68,9,29,0
American golden-plover,0,18,1,1,0,0,0,1,0,6,7,0,3,0,4,0,0,3,0,41,5,3,3,5,12,0,3,12,0,22,25,10,2,1,1,4,1,3,3,7,0,0,0,0,43,13,0,0,0,12,0,0,3,1,0,0,1,36,0,0,1,3,3,1,18,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Yellow-breasted chat,0,0,1,2,0,0,0,1,0,0,0,0,2,0,4,0,0,1,0,0,1,0,1,4,1,0,1,0,0,0,0,0,0,0,0,3,1,0,0,0,0,0,0,1,2,0,2,0,0,0,0,0,0,0,0,0,3,16,0,1,2,0,0,0,0,0,0
Yellow-crowned night heron,0,0,1,0,0,0,0,1,0,0,0,0,88,1,2,0,0,0,0,0,0,0,0,5,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0,2,0,0,0,0,12,0,0,0,0,0,3,0,0,0,0,0,1,27,0,0,0,11,0,0,0,0,0
Yellow-rumped warbler,0,2,0,2,0,2,0,33,5,2,4,0,50,0,20,0,0,0,0,18,1,0,0,4,9,0,2,8,0,6,3,9,0,1,0,30,2,0,1,26,0,1,0,0,84,4,4,0,12,4,0,0,1,4,1,0,3,22,0,4,10,0,3,15,5,0,0
Yellow-throated warbler,0,0,0,0,0,0,0,1,1,0,0,0,10,0,0,0,0,0,3,5,2,0,0,0,0,0,0,1,0,0,0,3,0,0,1,0,1,0,0,0,0,0,0,0,1,17,0,0,0,2,0,0,0,5,0,0,1,1,0,1,1,0,0,1,0,0,1


# analysis

### L1 normalization

In [74]:
df_norm = collisions_df.divide(collisions_df.sum(axis=1), axis=0)
df_norm.head(10)

,AB,AK,AL,AR,AS,AZ,BC,CA,CO,CT,DC,DE,FL,FN,GA,GU,HI,IA,ID,IL,IN,KS,KY,LA,MA,MB,MD,ME,MH,MI,MN,MO,MP,MS,MT,NC,ND,NE,NH,NJ,NL,NM,NS,NV,NY,OH,OK,ON,OR,PA,PR,QC,RI,SC,SD,SK,TN,TX,UM,UT,VA,VI,VT,WA,WI,WV,WY
American barn owl,0.0,0.000000,0.001748,0.003059,0.001311,0.003497,0.0,0.342220,0.015297,0.000000,0.006119,0.000000,0.017483,0.001748,0.020542,0.000437,0.174388,0.000000,0.003934,0.003497,0.000874,0.000437,0.002622,0.005245,0.000874,0.0,0.000437,0.000000,0.0,0.000874,0.000000,0.004808,0.000000,0.003497,0.000000,0.003934,0.000000,0.000437,0.000000,0.015734,0.0,0.000437,0.0,0.002185,0.074738,0.003497,0.002185,0.000000,0.086976,0.001311,0.000437,0.000000,0.000000,0.000000,0.000437,0.0,0.028846,0.050699,0.0,0.044580,0.002185,0.000000,0.000000,0.065997,0.000000,0.000000,0.000437
American black duck,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.051020,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010204,0.000000,0.000000,0.000000,0.000000,0.448980,0.0,0.000000,0.000000,0.0,0.000000,0.010204,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.040816,0.0,0.000000,0.0,0.000000,0.387755,0.010204,0.000000,0.000000,0.010204,0.010204,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.010204,0.000000,0.0,0.000000,0.010204,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
American coot,0.0,0.000000,0.000000,0.004773,0.000000,0.069212,0.0,0.226730,0.019093,0.000000,0.000000,0.000000,0.028640,0.000000,0.011933,0.000000,0.000000,0.000000,0.011933,0.054893,0.011933,0.002387,0.019093,0.004773,0.000000,0.0,0.000000,0.000000,0.0,0.002387,0.062053,0.023866,0.000000,0.002387,0.002387,0.002387,0.011933,0.011933,0.000000,0.002387,0.0,0.000000,0.0,0.052506,0.050119,0.002387,0.002387,0.000000,0.031026,0.007160,0.002387,0.000000,0.000000,0.002387,0.007160,0.0,0.021480,0.054893,0.0,0.116945,0.007160,0.000000,0.000000,0.045346,0.007160,0.002387,0.000000
American crow,0.0,0.002591,0.006477,0.002591,0.000000,0.001295,0.0,0.124352,0.011658,0.009067,0.010363,0.000000,0.019430,0.002591,0.020725,0.000000,0.000000,0.005181,0.002591,0.037565,0.012953,0.001295,0.029793,0.003886,0.016839,0.0,0.009067,0.022021,0.0,0.020725,0.032383,0.007772,0.000000,0.002591,0.003886,0.023316,0.003886,0.007772,0.009067,0.027202,0.0,0.001295,0.0,0.000000,0.075130,0.024611,0.003886,0.000000,0.076425,0.033679,0.000000,0.000000,0.002591,0.005181,0.001295,0.0,0.032383,0.072539,0.0,0.005181,0.019430,0.000000,0.018135,0.088083,0.011658,0.037565,0.000000
American golden-plover,0.0,0.052632,0.002924,0.002924,0.000000,0.000000,0.0,0.002924,0.000000,0.017544,0.020468,0.000000,0.008772,0.000000,0.011696,0.000000,0.000000,0.008772,0.000000,0.119883,0.014620,0.008772,0.008772,0.014620,0.035088,0.0,0.008772,0.035088,0.0,0.064327,0.073099,0.029240,0.005848,0.002924,0.002924,0.011696,0.002924,0.008772,0.008772,0.020468,0.0,0.000000,0.0,0.000000,0.125731,0.038012,0.000000,0.000000,0.000000,0.035088,0.000000,0.000000,0.008772,0.002924,0.000000,0.0,0.002924,0.105263,0.0,0.000000,0.002924,0.008772,0.008772,0.002924,0.052632,0.000000,0.000000
American goldfinch,0.0,0.000000,0.006289,0.000000,0.000000,0.000000,0.0,0.056604,0.012579,0.025157,0.006289,0.000000,0.012579,0.000000,0.025157,0.000000,0.000000,0.037736,0.012579,0.037736,0.025157,0.000000,0.044025,0.012579,0.006289,0.0,0.025157,0.018868,0.0,0.056604,0.012579,0.006289,0.000000,0.000000,0.006289,0.100629,0.012579,0.006289,0.025157,0.037736,0.0,0.000000,0.0,0.000000,0.088050,0.031447,0.006289,0.006289,0.025157,0.044025,0.000000,0.000000,0.006289,0.000000,0.012579,0.0,0.012579,0.018868,0.0,0.006289,0.006289,0.000000,0.000000,0.075472,0.025157,0.006289,0.000000
American kestrel,0.0,0.002566,0.002665,0.002566,0.000000,0.010561,0.0,0.087939,0.028523,0.011350,0.014508,0.000888,0.091887,0.000296,0.004244,0.000000,0.000000,0.003849,0.008291,0.109653,0.021911,0.002566,0.023885,0.002566,0.021220,0.0,0.014410,0.008291,0.0,0.024378,0.009278,0.020529,0.000000,0.001184,0.007402,

### find species of interest- high collision counts

In [75]:
df['SPECIES'].value_counts().head(20)

SPECIES
Mourning dove         16854
Killdeer              10963
Barn swallow          10946
American kestrel      10132
Horned lark            8969
Gulls                  7391
European starling      6809
Eastern meadowlark     5027
Rock pigeon            4594
Red-tailed hawk        4366
Sparrows               4314
Cliff swallow          3357
Western meadowlark     2816
Ring-billed gull       2419
American barn owl      2288
Canada goose           2192
Herring gull           2189
American robin         1906
Microbats              1709
Hawks                  1656
Name: count, dtype: int64

In [76]:
target_species = ['American kestrel', 'Horned lark', 'Herring gull']

In [77]:
for sp in target_species:
    print(df[df['SPECIES'] == sp].value_counts()[:10])

STATE  SPECIES         
IL     American kestrel    1111
FL     American kestrel     931
CA     American kestrel     891
NY     American kestrel     705
NJ     American kestrel     617
TX     American kestrel     577
OH     American kestrel     495
OR     American kestrel     408
PA     American kestrel     319
CO     American kestrel     289
Name: count, dtype: int64
STATE  SPECIES    
CO     Horned lark    4193
UT     Horned lark     545
MI     Horned lark     540
CA     Horned lark     441
MO     Horned lark     346
NY     Horned lark     232
MN     Horned lark     195
OH     Horned lark     159
MA     Horned lark     155
MT     Horned lark     149
Name: count, dtype: int64
STATE  SPECIES     
NY     Herring gull    945
MA     Herring gull    355
WI     Herring gull    145
NJ     Herring gull    107
CT     Herring gull    106
ME     Herring gull    102
OH     Herring gull     84
RI     Herring gull     50
MI     Herring gull     49
AK     Herring gull     31
Name: count, dtype: int64

### top 10 similar species by euclidean distance

In [78]:
dist = DistanceMetric.get_metric("euclidean")

for species in target_species:

    # get collision counts for target species
    target_species_collisions = df_norm.loc[species]

    #Generating distances from that species to all the others
    distances = dist.pairwise(df_norm, [target_species_collisions])[:,0]

    query_distances = list(zip(df_norm.index, distances))

    # top 10 most similar species
    print(f'{species}:\n')
    for similar_species, similar_state_score in sorted(query_distances, key=lambda x: x[1], reverse=False)[:10]:
        print(similar_species, similar_state_score)
        
    print('\n')



American kestrel:

American kestrel 0.0
Barn swallow 0.11329335771966093
Mallard 0.11670247886973122
Gulls 0.12276302762415234
Hawks 0.12336550987430592
Cooper's hawk 0.12439625484079551
Raccoon 0.1246040114218174
Sparrows 0.12575085047984874
Canada goose 0.13120805076769082
Song sparrow 0.14051700263843536


Horned lark:

Horned lark 0.0
Vesper sparrow 0.17624404469850732
Western meadowlark 0.23127539482487675
Great horned owl 0.24845207672958394
Eastern cottontail 0.26868105332339953
Gopher snake 0.3013038871806192
Desert cottontail 0.3628762009425579
White-tailed jackrabbit 0.36786579879721426
Cliff swallow 0.3836115493791307
White-crowned sparrow 0.38557303522227643


Herring gull:

Herring gull 0.0
Great black-backed gull 0.16861603110115578
Double-crested cormorant 0.18196647420005113
White-throated sparrow 0.23443042063208813
Veery 0.2492484996783436
Yellow-bellied sapsucker 0.2554593493858116
American woodcock 0.26421884602830087
Northern flicker 0.27684489672421114
Wood duck 0